# Slide 8: Final Model Stability Metrics

This notebook computes **per-fold cross-validation metrics** for the final champion model (Logistic Regression) at two operating thresholds:
- **t = 0.50** (F1-optimal baseline from Part B)
- **t = 0.27** (F2-optimal from Ablation 4, recall-biased for medical screening)

Unlike `cross_val_score` (which only supports sklearn's internal default threshold), this notebook manually iterates through CV folds, collects predicted probabilities, and applies both thresholds to compute fold-level classification metrics. This gives us proper **Mean ± Std Dev** stability estimates for each metric at each operating point.

**Prerequisites:**
- `logistic_regression_baseline.joblib`
- `X_train.csv`, `y_train.csv`, `X_test.csv`, `y_test.csv`

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    fbeta_score,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries imported.")

In [ ]:
# Load artifacts
artifact = joblib.load("logistic_regression_baseline.joblib")
baseline_pipeline = artifact["pipeline"]
le = artifact["label_encoder"]

baseline_params = baseline_pipeline.named_steps["logreg"].get_params()

print(f"Champion: Logistic Regression")
print(f"  C:            {baseline_params['C']}")
print(f"  class_weight: {baseline_params['class_weight']}")
print(f"  solver:       {baseline_params['solver']}")
print(f"  penalty:      {baseline_params['penalty']}")

# Load data
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train = le.transform(pd.read_csv("y_train.csv").values.ravel())
y_test = le.transform(pd.read_csv("y_test.csv").values.ravel())

print(f"\nX_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

## 2. Per-Fold Cross-Validation at Both Thresholds

For each of the 5 stratified folds:
1. Train the pipeline on the training portion
2. Predict probabilities on the held-out validation portion
3. Apply **both** thresholds (t=0.50 and t=0.27) to the same probabilities
4. Compute all metrics at each threshold

This ensures the stability comparison is fair — both thresholds see the exact same fold splits and probability scores.

In [ ]:
THRESHOLDS = {"t=0.50 (F1-optimal)": 0.50, "t=0.27 (F2-optimal)": 0.27}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Collect per-fold metrics for each threshold
fold_results = {name: [] for name in THRESHOLDS}

for fold_idx, (train_idx, val_idx) in enumerate(cv_strategy.split(X_train, y_train), 1):
    X_tr_fold = X_train.iloc[train_idx]
    y_tr_fold = y_train[train_idx]
    X_val_fold = X_train.iloc[val_idx]
    y_val_fold = y_train[val_idx]
    
    # Build and train a fresh pipeline with the same hyperparameters
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            C=baseline_params["C"],
            class_weight=baseline_params["class_weight"],
            solver=baseline_params["solver"],
            penalty=baseline_params["penalty"],
            max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ])
    pipe.fit(X_tr_fold, y_tr_fold)
    
    # Get probabilities (same for both thresholds)
    y_proba = pipe.predict_proba(X_val_fold)[:, 1]
    
    # Compute metrics at each threshold
    for name, t in THRESHOLDS.items():
        y_pred = (y_proba >= t).astype(int)
        
        fold_results[name].append({
            "Fold": fold_idx,
            "Accuracy": accuracy_score(y_val_fold, y_pred),
            "F1 (Yes)": f1_score(y_val_fold, y_pred),
            "F2 (Yes)": fbeta_score(y_val_fold, y_pred, beta=2),
            "Precision (Yes)": precision_score(y_val_fold, y_pred, zero_division=0),
            "Recall (Yes)": recall_score(y_val_fold, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_val_fold, y_proba),  # threshold-independent
            "Brier Score": brier_score_loss(y_val_fold, y_proba),  # threshold-independent
            "Log Loss": log_loss(y_val_fold, y_proba),  # threshold-independent
        })
    
    print(f"  Fold {fold_idx} complete.")

print("\nAll folds evaluated at both thresholds.")

## 3. Per-Fold Results (Detailed)

In [ ]:
for name, results in fold_results.items():
    df = pd.DataFrame(results)
    print(f"\n{'=' * 80}")
    print(f"  {name} — Per-Fold Results")
    print(f"{'=' * 80}")
    print(df.to_string(index=False, float_format="{:.4f}".format))

## 4. Final Stability Table (Slide 8 Deliverable)

This is the table for the slide: **Mean ± Std Dev** across 5 folds, at both thresholds.

In [ ]:
metrics_order = ["Accuracy", "F1 (Yes)", "F2 (Yes)", "Precision (Yes)", 
                 "Recall (Yes)", "ROC-AUC", "Brier Score", "Log Loss"]

stability_rows = []

for name, results in fold_results.items():
    df = pd.DataFrame(results)
    row = {"Threshold": name}
    for metric in metrics_order:
        mean = df[metric].mean()
        std = df[metric].std()
        row[metric] = f"{mean:.4f} +/- {std:.4f}"
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std
    stability_rows.append(row)

stability_df = pd.DataFrame(stability_rows)

# Display the clean version for the slide
display_cols = ["Threshold"] + metrics_order
print("=" * 120)
print("  FINAL MODEL STABILITY METRICS (5-Fold Stratified CV)")
print("  Model: Logistic Regression | L1 | C=0.0207 | class_weight={0:1, 1:5}")
print("=" * 120)
print()
print(stability_df[display_cols].to_string(index=False))

In [ ]:
# Also compute test set metrics at both thresholds for reference
# (these are NOT stability metrics — just single-point test performance)

# Fit on full training set
final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        C=baseline_params["C"],
        class_weight=baseline_params["class_weight"],
        solver=baseline_params["solver"],
        penalty=baseline_params["penalty"],
        max_iter=5000,
        random_state=RANDOM_STATE,
    )),
])
final_pipe.fit(X_train, y_train)
y_proba_test = final_pipe.predict_proba(X_test)[:, 1]

print("=" * 80)
print("  TEST SET PERFORMANCE (for reference, not stability)")
print("=" * 80)

for name, t in THRESHOLDS.items():
    y_pred = (y_proba_test >= t).astype(int)
    print(f"\n  {name}:")
    print(f"    Accuracy:       {accuracy_score(y_test, y_pred):.4f}")
    print(f"    F1 (Yes):       {f1_score(y_test, y_pred):.4f}")
    print(f"    F2 (Yes):       {fbeta_score(y_test, y_pred, beta=2):.4f}")
    print(f"    Precision (Yes):{precision_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"    Recall (Yes):   {recall_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"    ROC-AUC:        {roc_auc_score(y_test, y_proba_test):.4f}")
    print(f"    Brier Score:    {brier_score_loss(y_test, y_proba_test):.4f}")
    print(f"    Log Loss:       {log_loss(y_test, y_proba_test):.4f}")

## 5. Stability Visualisation

In [ ]:
# Visualise per-fold metric spread at both thresholds
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

plot_metrics = ["F1 (Yes)", "F2 (Yes)", "Precision (Yes)", "Recall (Yes)", "ROC-AUC", "Brier Score"]

for ax, metric in zip(axes, plot_metrics):
    for name, results in fold_results.items():
        df = pd.DataFrame(results)
        folds = df["Fold"]
        values = df[metric]
        short_label = name.split(" ")[0]  # "t=0.50" or "t=0.27"
        ax.plot(folds, values, "o-", linewidth=2, markersize=6, label=short_label)
        ax.axhline(values.mean(), linestyle="--", alpha=0.4)
    
    ax.set_xlabel("Fold")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Final Model Stability: Per-Fold Metrics at Both Thresholds", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("slide8_stability_metrics.png", dpi=150, bbox_inches="tight")
plt.close()
print("Plot saved: slide8_stability_metrics.png")

## 6. Interpretation for Slide 8

**Key points to highlight on the slide:**

1. **ROC-AUC, Brier Score, and Log Loss are identical** across both thresholds — these are threshold-independent metrics that reflect the model's intrinsic discrimination and calibration ability. The low std dev confirms stable probability outputs.

2. **F1 and Precision are higher at t=0.50**, while **F2 and Recall are higher at t=0.27** — this is expected and confirms the threshold is working as designed.

3. **Std dev at t=0.27 should be comparable or only slightly higher than t=0.50** — if so, the recall-biased threshold is not introducing instability; it's a stable policy shift.

4. **The final model choice** depends on the deployment context (Part E):
   - If you prioritise balanced classification → t=0.50 (higher F1)
   - If you prioritise catching heart disease cases → t=0.27 (higher F2, Recall)
   
   Both are valid configurations of the **same** stable model.

In [ ]:
# Save outputs
stability_df[display_cols].to_csv("slide8_stability_table.csv", index=False)
print("Saved: slide8_stability_table.csv")
print("Saved: slide8_stability_metrics.png")